# syft-crypto-python encryption benchmark

Pick a file size in GB, get encryption / decryption timings.

Uses the released `syft-crypto-python` from PyPI. Since `0.1.2b3` payloads over 4 GiB
are supported (the old `u32` envelope limit was lifted); on `0.1.2b2` or older, sizes
above ~4.29 GB fail with `ciphertext too large`.

In [ ]:
!uv pip install "syft-crypto-python==0.1.2b2"

from importlib.metadata import version

print("syft-crypto-python", version("syft-crypto-python"))

## Choose the file size

Fractional values work too (`0.5` = 500 MB). **Peak RAM is roughly 3× the payload** —
on a 32 GB machine stay at or below ~6.

In [ ]:
SIZE_GB = 5

In [ ]:
import syft_crypto_python as syc

# Sender (A) and recipient (B) — same key flow syft-client uses.
keys_a = syc.SyftRecoveryKey.generate().derive_keys()
keys_b = syc.SyftRecoveryKey.generate().derive_keys()
bundle_a = keys_a.to_public_bundle()
recipient_b = syc.EncryptionRecipient("b@test.org", keys_b.to_public_bundle())
print("identities ready")

In [ ]:
import gc
import hashlib
import os
import time

n_bytes = int(SIZE_GB * 1_000_000_000)

# 1 MiB random head + zeros: cheap to build, the cipher doesn't care about entropy.
head = os.urandom(min(1 << 20, n_bytes))
payload = head if n_bytes <= len(head) else head + bytes(n_bytes - len(head))
digest = hashlib.sha256(payload).hexdigest()
print(f"payload  : {n_bytes/1e9:.2f} GB")

t0 = time.perf_counter()
envelope = syc.encrypt_message("a@test.org", keys_a, [recipient_b], payload)
t_encrypt = time.perf_counter() - t0

del payload
gc.collect()

t0 = time.perf_counter()
parsed = syc.parse_envelope(envelope)
syc.verify_envelope_signature(parsed, bundle_a.identity_key_bytes)
decrypted = syc.decrypt_message("b@test.org", keys_b, bundle_a, parsed)
t_decrypt = time.perf_counter() - t0

ok = hashlib.sha256(decrypted).hexdigest() == digest
envelope_size = len(envelope)
del envelope, parsed, decrypted
gc.collect()

print(f"envelope : {envelope_size/1e9:.2f} GB")
print(f"encrypt  : {t_encrypt:7.2f} s   ({n_bytes/1e6/t_encrypt:7.1f} MB/s)")
print(f"decrypt  : {t_decrypt:7.2f} s   ({n_bytes/1e6/t_decrypt:7.1f} MB/s)")
print(f"integrity: {'PASS' if ok else 'FAIL'}")